In [49]:
import pandas as pd
import plotly.express as pep
import numpy as np
iris=pep.data.iris()

group分组

- by
    - 传入字符按标签分组
    - bool表达式/lambda
    - pipe
    - ['col',func] 混合分组
    - Grouper(key,level,freq,axis,sort) freq针对时间比如60s分组,
- axis 按行(0)或列(1)
- level 多层索引,则按一个或多个特定的层级进行拆分,支持数字层名
- as_index 数据分组聚合输出,默认返回带有租标签的对象多为索引
- sort 对分组排序
- group_keys 调用apply函数时,参数包含组名(废除)
- dropna 包含NA行/列删除
- observed 与分类相关,如果数据中有sepal_length'个分类没有数据就通过这个参数设置显示

返回DataFrameGroupBy对象,直接group['key']获取对应组的数据;分组好后可进行/agg/mean/sum聚合操作,transform转换操作,filter过滤操作,apply


分组对象
groups属性会生成一个字典,值为索引,可以通过keys获取分组名字;indices属性返回字典;

get_group([])获取相应分组

In [ ]:

iris.loc[iris.groupby('species').indices['versicolor']].head()

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
50,7.0,3.2,4.7,1.4,versicolor,2
51,6.4,3.2,4.5,1.5,versicolor,2
52,6.9,3.1,4.9,1.5,versicolor,2
53,5.5,2.3,4.0,1.3,versicolor,2
54,6.5,2.8,4.6,1.5,versicolor,2


In [17]:
def des(df,col_name):
    return pd.Series({'max':df[col_name].max(),'min':df[col_name].min()
            ,'mean':df[col_name].mean().round(2)})
iris.groupby('species').apply(des,'sepal_length')

/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1197/2980205711.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  iris.groupby('species').apply(des,'sepal_length')


,max,min,mean
species,,,
setosa,5.8,4.3,5.01
versicolor,7.0,4.9,5.94
virginica,7.9,4.9,6.59


In [20]:
iris.groupby('species').sepal_length.mean()

species
setosa        5.006
versicolor    5.936
virginica     6.588
Name: sepal_length, dtype: float64

In [ ]:
# loc[] 它不是函数,[]是索引器,__getitem__选取数据
# transform平铺数据
iris.loc[iris.groupby('species').transform('mean').sepal_length>4]

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
0,5.1,3.5,1.4,0.2,setosa,1
1,4.9,3.0,1.4,0.2,setosa,1
2,4.7,3.2,1.3,0.2,setosa,1
3,4.6,3.1,1.5,0.2,setosa,1
4,5.0,3.6,1.4,0.2,setosa,1
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica,3
146,6.3,2.5,5.0,1.9,virginica,3
147,6.5,3.0,5.2,2.0,virginica,3
148,6.2,3.4,5.4,2.3,virginica,3


In [ ]:
iris.groupby(by='species',group_keys=True,).agg({'sepal_length':['max','min'],'sepal_width':['mean','max']})

sepal_length      sepal_width     
                    max  min        mean  max
species                                      
setosa              5.8  4.3       3.418  4.4
versicolor          7.0  4.9       2.770  3.4
virginica           7.9  4.9       2.974  3.8

In [11]:
df = pd.DataFrame({
    '班级': ['一班', '一班', '二班', '二班'],
    '分数': [90, 85, 92, 88]
})

def apply_detail(x):
    print(x)

df.groupby(by='班级',group_keys=True).apply(apply_detail)
print('--'*20)
df.groupby(by='班级',group_keys=False).apply(apply_detail)

   班级  分数
0  一班  90
1  一班  85
   班级  分数
2  二班  92
3  二班  88
----------------------------------------
   班级  分数
0  一班  90
1  一班  85
   班级  分数
2  二班  92
3  二班  88


/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1368/1787703573.py:9: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1368/1787703573.py:11: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



""


In [12]:
df = pd.DataFrame({
    '性别': pd.Categorical(['男', '男', '男'], categories=['男', '女']),
    '分数': [80, 90, 100]
})

# 2. observed=True（默认，较新版本 Pandas 默认改为了 False，但我们手动对比看效果）
print("=== observed=True ===")
print(df.groupby('性别', observed=True).sum())

print("\n=== observed=False ===")
print(df.groupby('性别', observed=False).sum())

=== observed=True ===
     分数
性别     
男   270

=== observed=False ===
     分数
性别     
男   270
女     0


In [ ]:
df.groupby('性别',observed=True).transform('mean') #处理返回个数与输入个数相同

,分数
0,90.0
1,90.0
2,90.0


In [15]:
df.groupby('性别',observed=True).agg('mean')

,分数
性别,
男,90.0


In [19]:
iris.pipe(pd.DataFrame.groupby,'species')

In [ ]:
iris.convert_dtypes()
iris['species_id']=pd.Categorical(iris['species_id'],categories=[1,2,3])
iris.groupby(by=['species_id',lambda x:x<5],observed=False).count()

sepal_length  sepal_width  petal_length  petal_width  \
species                                                                  
setosa     False            45           45            45           45   
           True              5            5             5            5   
versicolor False            50           50            50           50   
virginica  False            50           50            50           50   

                  species_id  
species                       
setosa     False          45  
           True            5  
versicolor False          50  
virginica  False          50

In [27]:
iris.convert_dtypes().dtypes

sepal_length           Float64
sepal_width            Float64
petal_length           Float64
petal_width            Float64
species         string[python]
species_id               Int64
dtype: object

In [30]:
iris

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
0,5.1,3.5,1.4,0.2,NaN,1
1,4.9,3.0,1.4,0.2,NaN,1
2,4.7,3.2,1.3,0.2,NaN,1
3,4.6,3.1,1.5,0.2,NaN,1
4,5.0,3.6,1.4,0.2,NaN,1
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,NaN,3
146,6.3,2.5,5.0,1.9,NaN,3
147,6.5,3.0,5.2,2.0,NaN,3
148,6.2,3.4,5.4,2.3,NaN,3


聚合统计

- descibe
- mean
- count
- max/min
- median 中位数
- std 标准差
- var方差
- corr相关系数
- sem 标准误差
- prod 乘积
- cummax 每组累积最大值,从第一行向下走记录当前已走记录的最大的行
- cumsum
- mad 平均绝对偏差

In [47]:
species=iris.groupby('species')

species.cummax()[4:10]

,sepal_length,sepal_width,petal_length,petal_width,species_id
4,5.1,3.6,1.5,0.2,1
5,5.4,3.9,1.7,0.4,1
6,5.4,3.9,1.7,0.4,1
7,5.4,3.9,1.7,0.4,1
8,5.4,3.9,1.7,0.4,1
9,5.4,3.9,1.7,0.4,1


时间虚列采样resample
- rule (可加int表示区间)
    - D日
    - W
    - M 月末最后一天
    - MS 月初
    - T分钟
    - H 小时
- on 指定那列,如果索引为时间则可以不填
- close 采样区间是right/left
- label 聚合结果的标签取左边还是右边

In [ ]:
df=pd.DataFrame(data={'X':np.random.randint(1,20,(100,)),
                      'Date':pd.date_range('2021-01-01',freq='D',periods=100)})

In [68]:
df.resample('20D',on='Date',label='left').mean()

,X
Date,
2021-01-01,9.25
2021-01-21,8.85
2021-02-10,10.00
2021-03-02,8.95
2021-03-22,11.25


组内操作
- frist/last
- median/quantile 中位数和分位数
- diff 组累前后数据差

In [69]:
species.quantile(0.2)

,sepal_length,sepal_width,petal_length,petal_width,species_id
species,,,,,
setosa,4.7,3.1,1.3,0.20,1.0
versicolor,5.5,2.5,3.9,1.18,2.0
virginica,6.1,2.7,5.1,1.80,3.0


In [70]:
species.diff()

,sepal_length,sepal_width,petal_length,petal_width,species_id
0,NaN,NaN,NaN,NaN,NaN
1,-0.2,-0.5,0.0,0.0,0.0
2,-0.2,0.2,-0.1,0.0,0.0
3,-0.1,-0.1,0.2,0.0,0.0
4,0.4,0.5,-0.1,0.0,0.0
...,...,...,...,...,...
145,0.0,-0.3,-0.5,-0.2,0.0
146,-0.4,-0.5,-0.2,-0.4,0.0
147,0.2,0.5,0.2,0.1,0.0
148,-0.3,0.4,0.2,0.3,0.0


数据分箱
讲原始数据分成几个小区间,分箱可以大幅度减少观察误差

- cut 指定分界点对连续数据进行分箱
    -  labels 区间对映标签
    - qcut 指定区间数量对连续区间进行等宽分箱
> 配合group使用
> 它们返回categorical对象,有三个属性通过.cat获取
>   - .code
>   - .categories
>   - .label
> pd.get_dummies 变成区间0/1的变量

In [ ]:
df=pd.DataFrame(data={'X':np.random.randint(1,100,(100,)),
                      'Date':pd.date_range('2021-01-01',freq='D',periods=100)})

In [83]:
pd.qcut(df.X,q=4)

0      (36.75, 57.0]
1       (72.0, 99.0]
2     (0.999, 36.75]
3     (0.999, 36.75]
4       (57.0, 72.0]
           ...      
95      (72.0, 99.0]
96    (0.999, 36.75]
97     (36.75, 57.0]
98    (0.999, 36.75]
99     (36.75, 57.0]
Name: X, Length: 100, dtype: category
Categories (4, interval[float64, right]): [(0.999, 36.75] < (36.75, 57.0] < (57.0, 72.0] < (72.0, 99.0]]

In [84]:
df.X.groupby(pd.cut(x=df.X,bins=[0,45,75,100],labels=['A','B','C'])).count()

/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1197/1875636482.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.X.groupby(pd.cut(x=df.X,bins=[0,45,75,100],labels=['A','B','C'])).count()


X
A    36
B    44
C    20
Name: X, dtype: int64

In [ ]:
c=pd.qcut(df.X,q=4)
# print(c.cat.categories)
# print(c.cat.codes)
print(c.cat.categories.left[c.cat.codes]) #取出分箱区间左边值

Index([36.75,  72.0, 0.999, 0.999,  57.0, 0.999, 36.75, 36.75, 36.75, 0.999,
       0.999,  72.0,  57.0,  57.0, 0.999, 36.75,  57.0,  57.0,  57.0, 0.999,
       36.75,  57.0,  57.0, 36.75, 0.999,  57.0, 0.999, 36.75, 0.999,  72.0,
       0.999, 36.75, 36.75,  72.0, 36.75,  72.0, 0.999,  57.0,  57.0,  57.0,
       36.75,  57.0, 36.75,  72.0, 36.75,  72.0,  72.0,  57.0,  57.0, 36.75,
        72.0, 0.999,  57.0,  72.0,  57.0, 36.75,  72.0,  57.0,  72.0, 36.75,
        57.0,  72.0, 0.999,  72.0,  72.0,  72.0, 0.999,  57.0,  72.0,  57.0,
        72.0, 0.999,  57.0, 36.75, 36.75, 0.999, 36.75, 0.999, 0.999,  72.0,
       36.75, 36.75,  72.0, 0.999,  57.0,  57.0,  57.0, 36.75, 0.999,  72.0,
       36.75, 0.999, 0.999, 36.75,  72.0,  72.0, 0.999, 36.75, 0.999, 36.75],
      dtype='float64')
